# Assignment 2: Interacting with Web APIs - Weather data

Pull a 7-day hourly forecast for Pune (18.5204, 73.8567) from the free Open-Meteo API (no API key needed), clean it, summarize it by day and visualize it.

*Needs an internet connection.*

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

latitude = 18.5204
longitude = 73.8567

URL = 'https://api.open-meteo.com/v1/forecast'

response = requests.get(
    URL,
    params={
        'latitude': latitude,
        'longitude': longitude,
        'hourly': 'temperature_2m,relative_humidity_2m,precipitation',
        'forecast_days': 7,
        'timezone': 'Asia/Kolkata'
    },
    timeout=30
)

if response.status_code != 200:
    raise RuntimeError(f'API error {response.status_code}: {response.text}')

weather = response.json()

df = pd.DataFrame(weather['hourly'])

print(df.head())

## Extract and clean weather fields

In [ ]:
df = pd.DataFrame({
    'datetime': weather['hourly']['time'],
    'temperature_c': weather['hourly']['temperature_2m'],
    'humidity_pct': weather['hourly']['relative_humidity_2m'],
    'precipitation_mm': weather['hourly']['precipitation']
})

df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

for col in [
    'temperature_c',
    'humidity_pct',
    'precipitation_mm'
]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = (
    df
    .dropna(subset=['datetime', 'temperature_c'])
    .drop_duplicates()
    .sort_values('datetime')
    .reset_index(drop=True)
)

display(df.head())

## Summarize and model simple trends

Calculate averages and ranges, plus daily summaries. These are summaries of the forecast points returned by the API.

In [ ]:
display(
    df[['temperature_c', 'humidity_pct', 'precipitation_mm']].describe().T
)

df['date'] = df['datetime'].dt.date

daily = df.groupby('date', as_index=False).agg(
    avg_temp_c=('temperature_c', 'mean'),
    min_temp_c=('temperature_c', 'min'),
    max_temp_c=('temperature_c', 'max'),
    avg_humidity_pct=('humidity_pct', 'mean'),
    total_precipitation_mm=('precipitation_mm', 'sum')
)

display(daily)

## Visualize weather patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

df.plot(
    x='datetime',
    y='temperature_c',
    ax=axes[0,0],
    title='Forecast temperature (°C)',
    legend=False,
    color='tomato'
)

df.plot(
    x='datetime',
    y='humidity_pct',
    ax=axes[0,1],
    title='Humidity (%)',
    legend=False,
    color='royalblue'
)

df.plot(
    x='datetime',
    y='precipitation_mm',
    ax=axes[1,0],
    title='Precipitation (mm)',
    legend=False,
    color='slateblue'
)

axes[1,1].axis('off')

for ax in axes.flat:
    ax.set_xlabel('Date and time')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


plt.figure(figsize=(6, 4))

plt.scatter(
    df['temperature_c'],
    df['humidity_pct'],
    alpha=0.7
)

plt.xlabel('Temperature (°C)')
plt.ylabel('Humidity (%)')
plt.title('Temperature and Humidity')

plt.show()

print(
    'Temperature-humidity correlation:',
    round(df['temperature_c'].corr(df['humidity_pct']), 3)
)